In [1]:
import torch
import torch.nn as nn
import numpy as np

# ==========================
# User-specified problem
# ==========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Domain: [0,1] x [0,1]
# Define f(x,y) and boundary condition g(x,y)

def f_exact(x, y):
    """
    Right-hand side f(x,y) in -Δu = f.
    Example: choose an exact solution u(x,y) = sin(pi x) sin(pi y)
    then -Δu = 2*pi^2*sin(pi x)*sin(pi y)
    """
    return 2.0 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)

def g_bc(x, y):
    """
    Dirichlet boundary condition u(x,y) = g(x,y) on boundary.
    For the example exact solution u(x,y) = sin(pi x) sin(pi y),
    g is just that restricted to the boundary.
    """
    return np.sin(np.pi * x) * np.sin(np.pi * y)

# If you want a different problem, just change f_exact and g_bc accordingly.


# ==========================
# PINN model
# ==========================

class MLP(nn.Module):
    def __init__(self, in_dim=2, out_dim=1, hidden_dim=64, num_hidden=6):
        super().__init__()
        layers = []
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_hidden - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ==========================
# Sampling points
# ==========================

def sample_interior_points(N_int):
    # Uniform random points in (0,1)^2
    x = np.random.rand(N_int, 1)
    y = np.random.rand(N_int, 1)
    return x, y

def sample_boundary_points(N_bnd):
    # Sample boundary points on the square [0,1]^2
    # We split equally among four edges
    N_side = N_bnd // 4

    # x=0, y in [0,1]
    y1 = np.random.rand(N_side, 1)
    x1 = np.zeros_like(y1)

    # x=1, y in [0,1]
    y2 = np.random.rand(N_side, 1)
    x2 = np.ones_like(y2)

    # y=0, x in [0,1]
    x3 = np.random.rand(N_side, 1)
    y3 = np.zeros_like(x3)

    # y=1, x in [0,1]
    x4 = np.random.rand(N_side, 1)
    y4 = np.ones_like(x4)

    x = np.vstack([x1, x2, x3, x4])
    y = np.vstack([y1, y2, y3, y4])
    return x, y


# ==========================
# Loss terms
# ==========================

def pde_residual(model, x, y):
    """
    Compute PDE residual r(x,y) = -Δu(x,y) - f(x,y)
    using automatic differentiation.
    """
    x_t = torch.tensor(x, dtype=torch.float32, device=device, requires_grad=True)
    y_t = torch.tensor(y, dtype=torch.float32, device=device, requires_grad=True)

    inputs = torch.cat([x_t, y_t], dim=1)
    u = model(inputs)

    # First derivatives
    grads = torch.autograd.grad(
        u, inputs,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]
    u_x = grads[:, 0:1]
    u_y = grads[:, 1:2]

    # Second derivatives
    u_xx = torch.autograd.grad(
        u_x, inputs,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
        retain_graph=True
    )[0][:, 0:1]

    u_yy = torch.autograd.grad(
        u_y, inputs,
        grad_outputs=torch.ones_like(u_y),
        create_graph=True,
        retain_graph=True
    )[0][:, 1:2]

    laplace_u = u_xx + u_yy

    f_val = torch.tensor(f_exact(x, y), dtype=torch.float32, device=device).reshape(-1, 1)

    residual = -laplace_u - f_val
    return residual


def boundary_loss(model, x_b, y_b):
    x_t = torch.tensor(x_b, dtype=torch.float32, device=device)
    y_t = torch.tensor(y_b, dtype=torch.float32, device=device)
    inputs = torch.cat([x_t, y_t], dim=1)
    u_pred = model(inputs)
    u_true = torch.tensor(g_bc(x_b, y_b), dtype=torch.float32, device=device).reshape(-1, 1)
    return torch.mean((u_pred - u_true) ** 2)


# ==========================
# Training
# ==========================

def train_pinn(
    N_int=2000,
    N_bnd=400,
    num_epochs=5000,
    lr=1e-3,
    print_every=500
):
    model = MLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        # Sample points
        x_int, y_int = sample_interior_points(N_int)
        x_bnd, y_bnd = sample_boundary_points(N_bnd)

        # PDE residual loss
        res = pde_residual(model, x_int, y_int)
        loss_pde = torch.mean(res ** 2)

        # Boundary loss
        loss_bnd = boundary_loss(model, x_bnd, y_bnd)

        loss = loss_pde + loss_bnd

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % print_every == 0:
            print(f"Epoch {epoch}/{num_epochs} | Loss: {loss.item():.4e} | PDE: {loss_pde.item():.4e} | BND: {loss_bnd.item():.4e}")

    return model


# ==========================
# Evaluation on a grid
# ==========================

def evaluate_on_grid(model, nx=50, ny=50):
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)
    XY = np.stack([X.flatten(), Y.flatten()], axis=-1)
    XY_t = torch.tensor(XY, dtype=torch.float32, device=device)
    with torch.no_grad():
        U_pred = model(XY_t).cpu().numpy().reshape(ny, nx)
    return X, Y, U_pred


if __name__ == "__main__":
    # Train the PINN
    model = train_pinn(
        N_int=2000,
        N_bnd=400,
        num_epochs=5000,
        lr=1e-3,
        print_every=500
    )

    # Evaluate on grid
    X, Y, U_pred = evaluate_on_grid(model, nx=81, ny=81)

    # (Optional) Compare with exact solution if known
    U_exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    error = np.linalg.norm(U_pred - U_exact) / np.linalg.norm(U_exact)
    print(f"Relative L2 error vs exact solution: {error:.4e}")

    # You can visualize with matplotlib if you want:
    # import matplotlib.pyplot as plt
    # plt.figure()
    # cs = plt.contourf(X, Y, U_pred, levels=50, cmap="viridis")
    # plt.colorbar(cs)
    # plt.title("PINN solution u(x,y)")
    # plt.show()


Epoch 500/5000 | Loss: 4.4415e-02 | PDE: 3.3244e-02 | BND: 1.1171e-02
Epoch 1000/5000 | Loss: 3.1042e-02 | PDE: 2.1184e-02 | BND: 9.8580e-03
Epoch 1500/5000 | Loss: 3.0476e-02 | PDE: 2.1963e-02 | BND: 8.5132e-03
Epoch 2000/5000 | Loss: 9.8920e-03 | PDE: 1.6431e-03 | BND: 8.2489e-03
Epoch 2500/5000 | Loss: 2.2707e-02 | PDE: 1.5565e-02 | BND: 7.1428e-03
Epoch 3000/5000 | Loss: 1.0872e-02 | PDE: 3.5328e-03 | BND: 7.3390e-03
Epoch 3500/5000 | Loss: 1.0157e-02 | PDE: 3.9695e-03 | BND: 6.1880e-03
Epoch 4000/5000 | Loss: 6.7445e-03 | PDE: 7.3343e-04 | BND: 6.0111e-03
Epoch 4500/5000 | Loss: 1.4546e-02 | PDE: 8.6562e-03 | BND: 5.8896e-03
Epoch 5000/5000 | Loss: 2.7227e-02 | PDE: 2.1163e-02 | BND: 6.0632e-03
Relative L2 error vs exact solution: 7.3782e-02


In [ ]:
"""
PINN for 2D Poisson equation on [0,1] x [0,1]:

    -Δu(x,y) = f(x,y)   in Ω = (0,1)^2
     u(x,y)  = g(x,y)   on ∂Ω

You can change f_user(x,y) and g_user(x,y) below to define your own problem.
"""

import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# USER-SPECIFIED PROBLEM: edit these two functions
# ============================================================

def f_user(x, y):
    """
    Right-hand side f(x,y) in -Δu = f.

    Example: exact solution u(x,y) = sin(pi x) sin(pi y)
             => -Δu = 2*pi^2*sin(pi x)*sin(pi y)
    """
    return 2.0 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)


def g_user(x, y):
    """
    Dirichlet boundary condition u(x,y) = g(x,y) on ∂Ω.

    For the example above, g is just u restricted to the boundary.
    """
    return np.sin(np.pi * x) * np.sin(np.pi * y)


# ============================================================
# PINN model
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim=2, out_dim=1, hidden_dim=64, num_hidden=6):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(num_hidden - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# Sampling points in domain and on boundary
# ============================================================

def sample_interior_points(N_int):
    x = np.random.rand(N_int, 1)
    y = np.random.rand(N_int, 1)
    return x, y

def sample_boundary_points(N_bnd):
    N_side = N_bnd // 4

    # x=0
    y1 = np.random.rand(N_side, 1)
    x1 = np.zeros_like(y1)

    # x=1
    y2 = np.random.rand(N_side, 1)
    x2 = np.ones_like(y2)

    # y=0
    x3 = np.random.rand(N_side, 1)
    y3 = np.zeros_like(x3)

    # y=1
    x4 = np.random.rand(N_side, 1)
    y4 = np.ones_like(x4)

    x = np.vstack([x1, x2, x3, x4])
    y = np.vstack([y1, y2, y3, y4])
    return x, y


# ============================================================
# Loss terms: PDE residual and boundary loss
# ============================================================

def pde_residual(model, x, y):
    """
    r(x,y) = -Δu(x,y) - f(x,y)
    """
    x_t = torch.tensor(x, dtype=torch.float32, device=device, requires_grad=True)
    y_t = torch.tensor(y, dtype=torch.float32, device=device, requires_grad=True)

    inputs = torch.cat([x_t, y_t], dim=1)
    u = model(inputs)

    # First derivatives
    grads = torch.autograd.grad(
        u, inputs,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]
    u_x = grads[:, 0:1]
    u_y = grads[:, 1:2]

    # Second derivatives
    u_xx = torch.autograd.grad(
        u_x, inputs,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
        retain_graph=True
    )[0][:, 0:1]

    u_yy = torch.autograd.grad(
        u_y, inputs,
        grad_outputs=torch.ones_like(u_y),
        create_graph=True,
        retain_graph=True
    )[0][:, 1:2]

    laplace_u = u_xx + u_yy

    f_val = torch.tensor(f_user(x, y), dtype=torch.float32, device=device).reshape(-1, 1)
    residual = -laplace_u - f_val
    return residual


def boundary_loss(model, x_b, y_b):
    x_t = torch.tensor(x_b, dtype=torch.float32, device=device)
    y_t = torch.tensor(y_b, dtype=torch.float32, device=device)
    inputs = torch.cat([x_t, y_t], dim=1)
    u_pred = model(inputs)
    u_true = torch.tensor(g_user(x_b, y_b), dtype=torch.float32, device=device).reshape(-1, 1)
    return torch.mean((u_pred - u_true) ** 2)


# ============================================================
# Training loop
# ============================================================

def train_pinn(
    N_int=2000,
    N_bnd=400,
    num_epochs=5000,
    lr=1e-3,
    print_every=500
):
    model = MLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        x_int, y_int = sample_interior_points(N_int)
        x_bnd, y_bnd = sample_boundary_points(N_bnd)

        res = pde_residual(model, x_int, y_int)
        loss_pde = torch.mean(res ** 2)

        loss_bnd = boundary_loss(model, x_bnd, y_bnd)

        loss = loss_pde + loss_bnd

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % print_every == 0:
            print(
                f"Epoch {epoch}/{num_epochs} | "
                f"Loss: {loss.item():.4e} | "
                f"PDE: {loss_pde.item():.4e} | "
                f"BND: {loss_bnd.item():.4e}"
            )

    return model


# ============================================================
# Evaluation on a grid (and optional exact solution)
# ============================================================

def evaluate_on_grid(model, nx=50, ny=50):
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)
    XY = np.stack([X.flatten(), Y.flatten()], axis=-1)
    XY_t = torch.tensor(XY, dtype=torch.float32, device=device)
    with torch.no_grad():
        U_pred = model(XY_t).cpu().numpy().reshape(ny, nx)
    return X, Y, U_pred


if __name__ == "__main__":
    # Train PINN
    model = train_pinn(
        N_int=2000,
        N_bnd=400,
        num_epochs=5000,
        lr=1e-3,
        print_every=500
    )

    # Evaluate on grid
    X, Y, U_pred = evaluate_on_grid(model, nx=81, ny=81)

    # If you chose an exact solution (like the example), you can check error:
    U_exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    rel_L2 = np.linalg.norm(U_pred - U_exact) / np.linalg.norm(U_exact)
    print(f"Relative L2 error vs exact solution: {rel_L2:.4e}")

    # Optional: visualize
    # import matplotlib.pyplot as plt
    # plt.figure()
    # cs = plt.contourf(X, Y, U_pred, levels=50, cmap="viridis")
    # plt.colorbar(cs)
    # plt.title("PINN solution u(x,y)")
    # plt.show()


Epoch 500/5000 | Loss: 6.3054e-02 | PDE: 5.1119e-02 | BND: 1.1934e-02
Epoch 1000/5000 | Loss: 2.4531e-02 | PDE: 1.3355e-02 | BND: 1.1177e-02
Epoch 1500/5000 | Loss: 1.0102e-02 | PDE: 2.5773e-03 | BND: 7.5250e-03
